# 02 · Selectors, items and item loaders

Everything offline, on inline HTML: CSS/XPath details, typed items, and ItemLoader processors that clean data as it is extracted.

*Semua offline dengan HTML inline: detail CSS/XPath, item bertipe, dan processor ItemLoader.*

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.*

In [ ]:
// Packages / Paket
#r "nuget: Gravicode.ScrapyNet, 0.1.0"

// Before the packages are on nuget.org, point at a local build instead:
//   dotnet pack ScrapyNet.slnx -c Release -o artifacts
// and add this line above the #r lines:  #i "nuget: ../artifacts"

In [ ]:
using ScrapyNet;
using ScrapyNet.Loaders;

var html = """
    <div class="product" data-sku="A-1">
      <h2> Mechanical   Keyboard </h2>
      <span class="price">Rp 1.250.000</span>
      <ul class="specs"><li>Hot-swap</li><li>RGB</li><li>USB-C</li></ul>
      <a href="/p/a-1">details</a>
    </div>
    <div class="product" data-sku="B-2">
      <h2>Wireless Mouse</h2><span class="price">Rp 350.000</span>
      <ul class="specs"><li>Bluetooth</li></ul><a href="/p/b-2">details</a>
    </div>
    """;
var page = new HtmlResponse("https://shop.example/catalog", html);
display(page.Css("div.product::attr(data-sku)").GetAll());
display(page.Xpath("//div[has-class('product')]/h2/text()").GetAll());
display(page.Css("span.price::text").Re(@"[\d.]+"));

## Typed items with a loader

Processors turn messy strings into clean typed values: `Clean` collapses whitespace, `Price` understands both `1,299.00` and `1.250.000`.

*Processor mengubah teks berantakan menjadi nilai bertipe yang bersih.*

In [ ]:
public sealed class Product
{
    public string? Sku { get; set; }
    public string? Name { get; set; }
    public decimal Price { get; set; }
    public List<string> Specs { get; set; } = [];
    public string? Url { get; set; }
}

In [ ]:
var products = page.Css("div.product").Select(node =>
{
    var loader = new ItemLoader<Product>(node, response: page);
    loader.AddCss(p => p.Sku, "::attr(data-sku)");
    loader.AddCss(p => p.Name, "h2::text", Transforms.Clean);
    loader.AddCss(p => p.Price, "span.price::text", Transforms.Price);
    loader.AddCss(p => p.Specs, "ul.specs li::text");
    loader.Field(p => p.Url).Input(new MapCompose(Transforms.UrlJoin));
    loader.AddCss(p => p.Url, "a::attr(href)");
    return loader.LoadItem();
}).ToList();

display(products.Select(p => $"{p.Sku} | {p.Name} | {p.Price:N0} | {string.Join("/", p.Specs)} | {p.Url}"));